# Colab Paper-Run Notebook for `warp_optimization`

This notebook is meant for a heavier run than the smoke test. It clones the public repository, checks whether TensorFlow sees a GPU, runs one optimization bundle with paper-style settings, verifies the outputs, and compares the final parameters against the manuscript targets when available.

Recommended first serious runs: `domain=1, v=0.1, t=30.0` or `domain=2, v=0.1, t=30.0`.

In [ ]:
REPO_URL = "https://github.com/xys004/warp_bubble_optimization.git"
REPO_DIR = "warp_optimization"

!rm -rf "$REPO_DIR"
!git clone "$REPO_URL" "$REPO_DIR"
%cd $REPO_DIR
%pip install -r requirements.txt


In [ ]:
import tensorflow as tf

print('TensorFlow version:', tf.__version__)
gpus = tf.config.list_physical_devices('GPU')
print('Visible GPUs:', gpus)
if gpus:
    for gpu in gpus:
        try:
            tf.config.experimental.set_memory_growth(gpu, True)
        except Exception as exc:
            print('Could not set memory growth:', exc)
else:
    print('No GPU detected. The run will still work, but it may take much longer.')


In [ ]:
def fmt_num(value):
    text = f"{float(value):.3f}".rstrip('0').rstrip('.')
    return text.replace('.', 'p') if '.' in text else f"{text}p0"

RUN = {
    'domain': 2,
    'v': 0.1,
    't': 30.0,
    'seed': 47,
    'epochs': 2500,
    'n_xyz': 48,
    'pretrain_trials': 8,
    'pretrain_epochs': 60,
    'with_colorbar': True,
}

base = f"domain_{RUN['domain']}_v_{fmt_num(RUN['v'])}_t_{fmt_num(RUN['t'])}"
print('Planned base name:', base)
print(RUN)


In [ ]:
import subprocess

cmd = [
    'python', 'generate_run_bundle.py',
    '--domain', str(RUN['domain']),
    '--v', str(RUN['v']),
    '--t', str(RUN['t']),
    '--seed', str(RUN['seed']),
    '--epochs', str(RUN['epochs']),
    '--n-xyz', str(RUN['n_xyz']),
    '--pretrain-trials', str(RUN['pretrain_trials']),
    '--pretrain-epochs', str(RUN['pretrain_epochs']),
]
if RUN['with_colorbar']:
    cmd.append('--with-colorbar')
else:
    cmd.append('--no-colorbar')
print('Running:', ' '.join(cmd))
result = subprocess.run(cmd, check=False, text=True)
if result.returncode != 0:
    raise RuntimeError(f'generate_run_bundle.py failed with exit code {result.returncode}')


In [ ]:
from pathlib import Path

paths = sorted(Path('.').glob(f'{base}*'))
print('Generated files:')
for path in paths:
    print(' -', path.name)
if not paths:
    raise RuntimeError(f'No files were generated for base {base}')


In [ ]:
!python verify_outputs.py --base {base}


In [ ]:
!python compare_to_manuscript.py --base {base} --atol 0.10 --rtol 0.10


In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

field_patterns = [
    f'{base}_a*_XY_rho.png',
    f'{base}_a*_XY_WECmin.png',
    f'{base}_a*_XY_NECmin.png',
    f'{base}_a*_XY_DECmargin.png',
    f'{base}_a*_XY_SEC.png',
    f'{base}_a*_XZ_rho.png',
    f'{base}_a*_XZ_WECmin.png',
    f'{base}_a*_XZ_NECmin.png',
    f'{base}_a*_XZ_DECmargin.png',
    f'{base}_a*_XZ_SEC.png',
]
plot_files = []
for pattern in field_patterns:
    matches = sorted(Path('.').glob(pattern))
    if matches:
        plot_files.append(matches[0])

if not plot_files:
    raise RuntimeError(f'No field-map PNG files found for {base}')

fig, axes = plt.subplots(len(plot_files) // 2, 2, figsize=(14, 3.8 * (len(plot_files) // 2)))
axes = axes.ravel()
for ax, path in zip(axes, plot_files):
    ax.imshow(mpimg.imread(path))
    ax.set_title(path.name, fontsize=10)
    ax.axis('off')
for ax in axes[len(plot_files):]:
    ax.axis('off')
plt.tight_layout()
plt.show()


## Notes

- For a first validation, run the notebook exactly as-is for `domain=2, v=0.1, t=30.0`.
- Then rerun with `domain=1` to compare the single-shell target.
- If Colab falls back to CPU, keep the same settings but expect longer runtimes.
- The manuscript comparison checks final `A`, `B`, and `R0` against the published values when those targets are defined.